# Python Scripting — Assignment Solutions



## 1) Basic file operations and data processing

This section demonstrates:
- Creating and writing files
- Appending and reading files
- Counting words/lines
- Cleaning text and simple statistics
- Working with CSV (read/write) and basic aggregations


In [1]:
import os
from pathlib import Path

# Create a working folder (inside Colab or local runtime)
WORKDIR = Path("assignment_data")
WORKDIR.mkdir(exist_ok=True)

text_file = WORKDIR / "sample.txt"

# 1) Write a new file
sample_text = (
    "Python is a powerful language.\n"
    "It is used for scripting, automation, data analysis, and web development.\n"
    "File handling in Python is simple and effective.\n"
)

text_file.write_text(sample_text, encoding="utf-8")

print("Created:", text_file)
print("File size (bytes):", text_file.stat().st_size)


Created: assignment_data/sample.txt
File size (bytes): 154


In [2]:
# 2) Append to the same file
with open(text_file, "a", encoding="utf-8") as f:
    f.write("\nThis line was appended later.\n")

# 3) Read and display the content
content = text_file.read_text(encoding="utf-8")
print(content)


Python is a powerful language.
It is used for scripting, automation, data analysis, and web development.
File handling in Python is simple and effective.

This line was appended later.



In [4]:
import re
from collections import Counter

# 4) Data processing: count lines, words, and most common words
lines = content.strip().splitlines()
words = re.findall(r"[A-Za-z']+", content.lower())  # basic word tokenizer

print("Line count:", len(lines))
print("Word count:", len(words))

word_freq = Counter(words)
print("\nTop 10 most common words:")
for w, c in word_freq.most_common(10):
    print(f"{w:<12} {c}")


Line count: 5
Word count: 29

Top 10 most common words:
is           3
python       2
and          2
a            1
powerful     1
language     1
it           1
used         1
for          1
scripting    1


In [5]:
# 5) Extract emails/phones example (data extraction from text)
# (Demonstration only: we'll add an example line containing email/phone)

extra = "\nContact: support@example.com, Phone: +91-98765-43210\n"
text_file.write_text(content + extra, encoding="utf-8")

content2 = text_file.read_text(encoding="utf-8")

emails = re.findall(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", content2)
phones = re.findall(r"(?:\+?\d{1,3}[-\s]?)?\d{5}[-\s]?\d{5}", content2)

print("Emails found:", emails)
print("Phones found:", phones)


Emails found: ['support@example.com']
Phones found: ['+91-98765-43210']


### CSV data processing example

We will:
- Create a CSV file with sample sales data
- Read it back
- Compute totals and a group-by summary (total sales per product)


In [6]:
import csv

csv_file = WORKDIR / "sales.csv"

rows = [
    ["date", "product", "qty", "price"],
    ["2026-03-01", "Shampoo", 2, 299],
    ["2026-03-01", "Conditioner", 1, 349],
    ["2026-03-02", "Shampoo", 1, 299],
    ["2026-03-03", "Hair Serum", 3, 499],
    ["2026-03-03", "Conditioner", 2, 349],
]

with open(csv_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print("Created:", csv_file)


Created: assignment_data/sales.csv


In [7]:
from collections import defaultdict

# Read CSV and compute totals per product + grand total
sales_by_product = defaultdict(int)
grand_total = 0

with open(csv_file, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        qty = int(r["qty"])
        price = int(r["price"])
        total = qty * price
        sales_by_product[r["product"]] += total
        grand_total += total

print("Grand total sales:", grand_total)
print("\nTotal sales by product:")
for product, total in sorted(sales_by_product.items(), key=lambda x: x[1], reverse=True):
    print(f"{product:<12} {total}")


Grand total sales: 3441

Total sales by product:
Hair Serum   1497
Conditioner  1047
Shampoo      897


## 2) Simple web scraper to extract data from a website

### Option A: `requests` + `BeautifulSoup`

This approach:
- Downloads the HTML page using `requests`
- Parses it using BeautifulSoup
- Extracts structured fields (page title and a few links)



In [ ]:
# If BeautifulSoup isn't installed in your runtime, uncomment:
# !pip -q install beautifulsoup4 requests


In [9]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def scrape_page_titles(url: str, limit: int = 10):
    """Fetch a web page and extract page title + first `limit` links."""
    headers = {"User-Agent": "Mozilla/5.0 (compatible; AssignmentBot/1.0)"}
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    page_title = soup.title.get_text(strip=True) if soup.title else "(no title)"
    links = []
    for a in soup.select("a[href]")[:limit]:
        text = a.get_text(" ", strip=True) or "(no text)"
        href = urljoin(url, a["href"])
        links.append({"text": text, "url": href})

    return {"url": url, "title": page_title, "links": links}

# Example usage (safe demo site):
result = scrape_page_titles("https://www.wikipedia.org/", limit=10)
result


{'url': 'https://www.wikipedia.org/',
 'title': 'Wikipedia',
 'links': [{'text': 'English 7,141,000+ articles',
   'url': 'https://en.wikipedia.org/'},
  {'text': 'æ\x97¥æ\x9c¬èª\x9e 1,491,000+ è¨\x98äº\x8b',
   'url': 'https://ja.wikipedia.org/'},
  {'text': 'Deutsch 3.099.000+ Artikel', 'url': 'https://de.wikipedia.org/'},
  {'text': 'Ð\xa0Ñ\x83Ñ\x81Ñ\x81ÐºÐ¸Ð¹ 2Â\xa0087Â\xa0000+ Ñ\x81Ñ\x82Ð°Ñ\x82ÐµÐ¹',
   'url': 'https://ru.wikipedia.org/'},
  {'text': 'FranÃ§ais 2â\x80¯740â\x80¯000+ articles',
   'url': 'https://fr.wikipedia.org/'},
  {'text': 'EspaÃ±ol 2.095.000+ artÃ\xadculos',
   'url': 'https://es.wikipedia.org/'},
  {'text': 'Italiano 1.957.000+ voci', 'url': 'https://it.wikipedia.org/'},
  {'text': 'ä¸\xadæ\x96\x87 1,524,000+ æ\x9d¡ç\x9b® / æ¢\x9dç\x9b®',
   'url': 'https://zh.wikipedia.org/'},
  {'text': 'PortuguÃªs 1.165.000+ artigos',
   'url': 'https://pt.wikipedia.org/'},
  {'text': 'Polski 1Â\xa0685Â\xa0000+ haseÅ\x82',
   'url': 'https://pl.wikipedia.org/'}]}

### Option B (Reproducible demo without internet): parse saved HTML

If internet access is restricted:
- Using a saved HTML string (or reading from a `.html` file)
- Parsing it with BeautifulSoup
- Extracting fields just like a real website


In [10]:
from bs4 import BeautifulSoup

html = """
<!doctype html>
<html>
  <head><title>Demo Site</title></head>
  <body>
    <h1>News</h1>
    <ul>
      <li><a href="/post1">Python Tips</a></li>
      <li><a href="/post2">Web Scraping Basics</a></li>
      <li><a href="/post3">File Handling</a></li>
    </ul>
    <table id="prices">
      <tr><th>Item</th><th>Price</th></tr>
      <tr><td>Book</td><td>499</td></tr>
      <tr><td>Course</td><td>1999</td></tr>
    </table>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")

title = soup.title.get_text(strip=True)
posts = [a.get_text(strip=True) for a in soup.select("ul li a")]
prices = [
    (row.select_one("td:nth-child(1)").get_text(strip=True),
     row.select_one("td:nth-child(2)").get_text(strip=True))
    for row in soup.select("#prices tr")[1:]
]

print("Title:", title)
print("Posts:", posts)
print("Prices:", prices)


Title: Demo Site
Posts: ['Python Tips', 'Web Scraping Basics', 'File Handling']
Prices: [('Book', '499'), ('Course', '1999')]
